In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

# Move to repo root so all paths work
_p = Path.cwd()
while not (_p / "demos").exists() and _p != _p.parent:
    _p = _p.parent
if (_p / "demos").exists():
    os.chdir(_p)

load_dotenv()
DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


# Images

You already know what to do with text: summarize it, answer questions about it, extract data from it. Images, audio, and video are just ways of **getting to text and structured data.**

## Structured output

Send an image to an LLM and get back structured data — fields you can sort, filter, and verify. Not prose. This is the pattern for everything else in the workshop.


In [ ]:
from IPython.display import Image
Image(filename="data/car.jpg", width=500)


**`vision-llm/structured.py`** — Send one image to an LLM, get structured Pydantic data back


In [ ]:
from pathlib import Path

from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

MODEL = "openai:gpt-4o-mini"
DATA = Path("data")

class Vehicle(BaseModel):
    make: str = Field(description="Vehicle manufacturer (e.g., Toyota, Ford)")
    model: str = Field(description="Vehicle model name (e.g., Camry, F-150)")
    color: str = Field(description="Primary color of the vehicle")
    year_estimate: int = Field(description="Estimated model year (best guess)")
    vehicle_type: str = Field(description="Type: sedan, SUV, truck, van, motorcycle, etc.")
    confidence: float = Field(description="Your confidence in this identification, 0.0 to 1.0")

agent = Agent(MODEL, output_type=Vehicle)
result = agent.run_sync([
    "Analyze the vehicle in this image. Fill in all fields accurately.",
    BinaryContent(data=(DATA / "car.jpg").read_bytes(), media_type="image/jpeg"),
])
result.output


Notice the Pydantic model: each field has a name, a type, and a description. The LLM fills in the fields. If it's wrong, you can see *which* field is wrong.

## Batch processing

Same thing, whole folder. Out comes a CSV.


**`vision-llm/batch.py`** — Process a folder of images into structured data -> DataFrame -> CSV


In [ ]:
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

MODEL = "openai:gpt-4o-mini"
DATA = Path("data")

class Vehicle(BaseModel):
    make: str = Field(description="Vehicle manufacturer")
    model: str = Field(description="Vehicle model name")
    color: str = Field(description="Primary color")
    year_estimate: int = Field(description="Estimated model year")
    vehicle_type: str = Field(description="sedan, SUV, truck, van, etc.")
    confidence: float = Field(description="Confidence in identification, 0.0 to 1.0")

agent = Agent(MODEL, output_type=Vehicle)
rows = []
for image_path in sorted((DATA / "cars").glob("*.jpg")):
    result = agent.run_sync([
        "Analyze the vehicle in this image. Fill in all fields.",
        BinaryContent(data=image_path.read_bytes(), media_type="image/jpeg"),
    ])
    rows.append({"filename": image_path.name, **result.output.model_dump()})

df = pd.DataFrame(rows)
output = Path("outputs") / "cars_analysis.csv"
output.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output, index=False)
df


Open the output CSV. Spot-check a few rows against the source images. Does the make match what you see? Does the color? That's verification — not trusting the model, checking its work.

## Swap providers

Same structured task, different LLM provider. Change one string.


**`vision-llm/providers.py`** — Same structured-output task with different LLM providers


In [ ]:
from pathlib import Path

from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")
image_data = (DATA / "sky.jpg").read_bytes()

class ImageDescription(BaseModel):
    subject: str = Field(description="Main subject of the image")
    setting: str = Field(description="Where the image appears to be taken")
    mood: str = Field(description="Overall mood or feeling of the image")
    details: list[str] = Field(description="3-5 notable details")

models = [
    "openai:gpt-4o-mini",
    "google-gla:gemini-2.5-flash",
    # "anthropic:claude-3-5-haiku-latest",
    # "ollama:qwen2-vl",
]
for model in models:
    agent = Agent(model, output_type=ImageDescription)
    result = agent.run_sync([
        "Describe this image. Fill in all fields.",
        BinaryContent(data=image_data, media_type="image/jpeg"),
    ])
    print(f"--- {model} ---")
    print(result.output)


OpenAI, Google, Anthropic, Ollama — the code is identical except for the model name. Pick whichever fits your newsroom's budget, privacy needs, or existing accounts.
